# Flux Dev Photo Generator — Colab T4

**Instagram-quality portrait photos** with Flux Dev fp8 + IPAdapter FaceID + Style Transfer.

| Workflow | Input | Output |
|----------|-------|--------|
| `photo_flux_instagram` | Reference face photos + text | Consistent FaceID portraits |
| `photo_flux_img2img` | Reference image + text | Style/composition transfer |

**Run cells 1-6 in order**, then open the tunnel URL.

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ComfyUI + Custom Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Already cloned"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Custom nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Already cloned"

!pip install insightface onnxruntime -q
!pip install -r ComfyUI_IPAdapter_plus/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nInstalled!")

In [ ]:
#@title 3. Download Models (~18 GB)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/checkpoints", exist_ok=True)
os.makedirs(f"{M}/clip_vision", exist_ok=True)
os.makedirs(f"{M}/ipadapter", exist_ok=True)

files = {
    # Flux Dev fp8 (~11.5 GB)
    f"{M}/checkpoints/flux1-dev-fp8.safetensors":
        "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    # CLIP Vision H14 (~3.9 GB)
    f"{M}/clip_vision/model.safetensors":
        "https://huggingface.co/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/open_clip_pytorch_model.safetensors",
    # IPAdapter FaceID PlusV2 SDXL (~0.9 GB)
    f"{M}/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin":
        "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin",
    # IPAdapter Flux (style transfer) (~1.5 GB)
    f"{M}/ipadapter/ip-adapter_flux.safetensors":
        "https://huggingface.co/InstantX/FLUX.1-dev-IP-Adapter/resolve/main/ip-adapter.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Already exists: {os.path.basename(dest)}")
    else:
        print(f"\nDownloading {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nAll models ready!")
!du -sh {M}/checkpoints/* {M}/clip_vision/* {M}/ipadapter/*

In [ ]:
#@title 4. Download Workflows from Gist
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

# Download Colab-specific version (fixed paths)
!wget -q -O "{WF_DIR}/photo_flux_instagram.json" "{RAW}/photo_flux_instagram_colab.json"
print("Downloaded: photo_flux_instagram (Colab paths)")

# Download img2img workflow
!wget -q -O "{WF_DIR}/photo_flux_img2img.json" "{RAW}/photo_flux_img2img.json"
print("Downloaded: photo_flux_img2img")

print(f"\n2 workflows saved to {WF_DIR}")

In [ ]:
#@title 5. Upload Reference Photos
#@markdown Upload face reference photos. They will be available in the LoadImage nodes.
#@markdown The workflow has 3 folder slots: **portrait**, **hall**, **street**.

from google.colab import files
import os

# Create ref folders matching workflow structure
INPUT = "/content/ComfyUI/input"
for folder in ["refs/portrait", "refs/hall", "refs/street"]:
    os.makedirs(f"{INPUT}/{folder}", exist_ok=True)

target = "refs/portrait" #@param ["refs/portrait", "refs/hall", "refs/street"]

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT, target, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Saved: {path}")

print(f"\nFiles in {target}: {os.listdir(os.path.join(INPUT, target))}")

In [ ]:
#@title 6. Launch ComfyUI
import subprocess, time, re

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Starting ComfyUI... (30s)")
time.sleep(30)

# Cloudflare tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI ready! Open: {url}")
    print(f"{'='*60}")
    print("\nLoad workflow: Menu -> Load -> photo_flux_instagram")
else:
    print("Tunnel URL not found. ComfyUI running on port 8188.")

In [ ]:
#@title 7. Save Outputs to Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/flux"
os.makedirs(dst, exist_ok=True)

images = glob.glob(f"{src}/*.png") + glob.glob(f"{src}/*.jpg")
if not images:
    print("No outputs found yet. Generate some images first!")
else:
    for img in images:
        shutil.copy2(img, dst)
        print(f"Copied: {os.path.basename(img)}")
    print(f"\n{len(images)} files saved to {dst}")

---
## Usage

### Workflow 1: `photo_flux_instagram` — FaceID Portraits
1. Open the tunnel URL
2. Load **photo_flux_instagram** workflow
3. The workflow loads reference face photos from 3 folders (portrait/hall/street)
4. Use **ImageMaskSwitch** to select folder: 1=portrait, 2=hall, 3=street
5. Edit prompt and hit **Queue Prompt**

### Workflow 2: `photo_flux_img2img` — Style Transfer
1. Load **photo_flux_img2img** workflow
2. Upload a reference image (the style/composition source)
3. Edit the prompt to describe desired output
4. IPAdapter transfers the style while the prompt guides content
5. Adjust `denoise` in KSampler (0.65 = moderate change, 0.9 = strong change)

### Using a LoRA
1. Run cell 8 to download your LoRA
2. In ComfyUI, add a **LoraLoader** node between CheckpointLoader and KSampler
3. Select your LoRA file
4. Use the trigger word in your prompt (e.g., "a photo of ohwx person")
5. LoRA strength 0.7-1.0 works best for faces

### Folder Paths (already configured for Colab)
- `/content/ComfyUI/input/refs/portrait`
- `/content/ComfyUI/input/refs/hall`
- `/content/ComfyUI/input/refs/street`

### Output
- Resolution: 864x1080 (portrait)
- Prefix: `instagram_flux` / `flux_img2img`
- Files saved to `/content/ComfyUI/output/`

---
## Usage

1. Open the tunnel URL
2. Load **photo_flux_instagram** workflow
3. The workflow loads reference face photos from 3 folders (portrait/hall/street)
4. Use **ImageMaskSwitch** to select folder: 1=portrait, 2=hall, 3=street
5. Edit prompt and hit **Queue Prompt**

### Folder Paths on Colab
Update the `LoadImagesFromFolderKJ` nodes if paths differ:
- `/content/ComfyUI/input/refs/portrait`
- `/content/ComfyUI/input/refs/hall`
- `/content/ComfyUI/input/refs/street`

### Output
- Resolution: 864x1080 (portrait)
- Prefix: `instagram_flux`
- Files saved to `/content/ComfyUI/output/`